### Read dataset

In [18]:
import pandas as pd
df = pd.read_csv("tp1_dataset.csv", usecols=["book_desc", "genres"])

print(df.shape)
print(df.dtypes)

(54301, 2)
book_desc    str
genres       str
dtype: object


### Analyze description texts

In [3]:
print(df["book_desc"].head(10))
print(df["book_desc"].describe())

0    Winning will make you famous. Losing means cer...
1    There is a door at the end of a silent corrido...
2    The unforgettable novel of a childhood in a sl...
3    «È cosa ormai risaputa che a uno scapolo in po...
4    About three things I was absolutely positive.F...
5    Trying to make sense of the horrors of World W...
6    Journeys to the end of the world, fantastic cr...
7    مزرعة الحيوانات هي رائعة جورج أورويل الخالدة.....
8    Gone with the Wind is a novel written by Marga...
9    لجزء الثالث من ملحمة جيه أر أر تولكين الرائعة ...
Name: book_desc, dtype: str
count                                                 52970
unique                                                51781
top       هذه هي طبعة "دار الفكر - بيروت" وهي آخر طبعة ع...
freq                                                     38
Name: book_desc, dtype: object


conclusions from this code block:
- only found 52970 descriptions over 54301 rows, this means there are some NA (missing) values
- identified multiple languages and writing systems, as for example Abjad (Arabic)
- unique values (51781) is lower than all texts found (52970), this means there are some duplicated entries

In [4]:
repeated = df.groupby(["book_desc"]).size().reset_index(name="count")
repeated = repeated[repeated["count"] > 1]

print(repeated)

                                               book_desc  count
238    \r\n\r\n\r\nBased on the true story of a forgo...      2
557    \r\n\r\nWar has erupted in the Banished Lands ...      2
926     Until now, he's existed only in her dreams -b...      2
956     milk and honey  is a collection of poetry and...      2
1135   "As Gregor Samsa awoke one morning from uneasy...      2
...                                                  ...    ...
51258  “Did you ever hear the Tragedy of Darth Plague...      2
51304  “Her knees were weak and if not for that marbl...      2
51461  “Our Dragon doesn’t eat the girls he takes, no...      2
51640  „Als ich zum ersten Mal in deine Augen sah, wu...      2
51649  „Dievų miškas“ — memuarinis lietuvių rašytojo ...      2

[926 rows x 2 columns]


conclusions from this code block:
- found entries with carriage returns and line feeds, this don't add useful semantic information, may harm the tokenization process or produce slightly different embeddings for the same sentence.

### Cleanup and normalize dataset

In [19]:
print(df.shape)

import re
import unicodedata

def clean_text(text):
    # if the text is NaN (missing value), return it as is to avoid processing
    if pd.isna(text):
        return text
    text = str(text)
    # remove carriage returns and line feeds
    text = re.sub(r"(\\[rn]|[\r\n])+", " ", text)
    # replace multiple whitespace characters with a single space and trim leading/trailing whitespace
    text = re.sub(r"\s+", " ", text).strip()
    # if the resulting text is empty after cleaning, return None to indicate missing value
    if text == "":
        return None
    # normalize to unicode to ensure consistent representation of characters
    text = unicodedata.normalize("NFKC", text)
    return text

df["book_desc_clean"] = df["book_desc"].apply(clean_text)
# drop null values from descriptions and genres, as they cannot be used for training the model
df = df.dropna(subset=["book_desc_clean", "genres"])
# drop duplicated descriptions
df = df.drop_duplicates(subset="book_desc_clean", keep="last")
# reset index after dropping rows to avoid issues with indexing later on
df = df.reset_index(drop=True)

print(df.shape)

(54301, 2)
(49002, 3)


Pre-processing performed:
- For each description:
    - if there is a value, removed all the carriage returns, line feeds and additional whitespaces,
    - if the text is not empty then normalize text fixing any Unicode inconsistencies, helpful specially when dealing with multiple languages and writing systems
- Remove all rows with missing values on description or genres
- Remove duplicated descriptions, keeping the last of the duplicated, assuming the last one would be the most updated record (just for learning purposes)

## Language issue
descriptions in multiple languages and we dont have a column identifying the language to filter


In [20]:
from fast_langdetect import detect

def detect_language(text):
    result = detect(str(text), model='lite', k=1)
    return result[0]["lang"], result[0]["score"]

df[["lang", "lang_confidence"]] = df["book_desc_clean"].apply(lambda x: pd.Series(detect_language(x)))

lang_confidence_threshold = 0.7
df["high_conf"] = df["lang_confidence"] > lang_confidence_threshold

languages_stats = df.groupby("lang").agg(
    high_confidence=("high_conf", "sum"),
    total=("lang", "size")
)
print(languages_stats.to_string())

      high_confidence  total
lang                        
af                  0      1
ar                680    695
arz                 1      2
azb                 0      1
bg                 71     77
bn                 26     26
bs                  0      1
ca                  8     12
ceb                 0      3
cs                105    108
da                 41     68
de                574    641
el                 94     95
en              39383  42674
eo                  0      2
es                828    891
et                 16     21
fa                126    135
fi                 90     94
fr                555    594
fy                  0      1
gl                  1      3
hi                  5      6
hr                  0     32
hu                 26     29
hy                  1      1
id                201    337
is                  3      3
it                391    433
ja                 54     55
ka                 39     39
ko                  5      6
la            

Conclusions from this code block:
- Besides english we dont have a large amount of descriptions for each language
- For english, we have 37517 records with language confidence higher than 80%

So we have two options to vectorize the descriptions:
1) use TF-IDF vectorizer, that requires some additional NLP strategies to reduce noise on the data while keeping the context, like stopwords removal and lemmatization, but this approach is only viable for the the books that we are almost certain that are in english, as it is the only language that we have acceptable amount of data, 39383 records.
2) Use a pretrained multilingual embeddings model (as Sentence-BERT) to generate a fixed-size vector that supports multiple languages and preserve semantics, as it is almost language agnostic, we can use all the descriptions, 49002


# Vectorize texts

### Approach 1 - Use tf-idf together with more preprocessing to vectorize descriptions

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

df_english = df[(df["lang"] == "en") & (df["lang_confidence"] > lang_confidence_threshold)]

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
vectorizer = TfidfVectorizer(max_features=5000)

def preprocess(text):
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return " ".join(tokens)

lemmas = df_english["book_desc_clean"].apply(lambda x: preprocess(str(x)))

X = vectorizer.fit_transform(lemmas).toarray()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\z004n7mh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\z004n7mh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\z004n7mh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\z004n7mh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


### Approach 2 - Use transformer to vectorize descriptions

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

X_text = df['book_desc_clean'].tolist()

transformer = SentenceTransformer('distiluse-base-multilingual-cased-v2')
X = transformer.encode(
    X_text,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    device=device
)

np.save("book_desc_embeddings.npy", X)

Using device: cuda


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8233.33it/s]


## multi-label binarization of genres

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

def preprocess_genres(approach = ""):
    if approach == "":
        approach = input("Choose approach (1 for TF-IDF, 2 for Sentence Transformers): ")

    genders = df_english["genres"].apply(lambda x: str(x).split("|")) if approach == "1" else df["genres"].apply(lambda x: str(x).split("|"))

    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(genders)
    
    return Y, mlb

Y, mlb = preprocess_genres()

print(Y.shape)
print(mlb.classes_)



(49002, 864)
['10th Century' '11th Century' '12th Century' '13th Century'
 '14th Century' '15th Century' '16th Century' '17th Century'
 '18th Century' '1917' '19th Century' '1st Grade' '20th Century'
 '21st Century' '2nd Grade' '40k' 'Abandoned' 'Abuse' 'Academia'
 'Academic' 'Academics' 'Action' 'Activism' 'Adaptations' 'Adolescence'
 'Adoption' 'Adult' 'Adult Fiction' 'Adventure' 'Aeroplanes' 'Africa'
 'African American' 'African American Literature'
 'African American Romance' 'African Literature' 'Agriculture' 'Aircraft'
 'Albanian Literature' 'Alchemy' 'Alcohol' 'Algeria' 'Algorithms' 'Aliens'
 'Alternate History' 'Alternative Medicine' 'Amazon' 'American'
 'American Civil War' 'American Fiction' 'American History'
 'American Revolution' 'American Revolutionary War' 'Americana' 'Amish'
 'Amish Fiction' 'Anarchism' 'Ancient' 'Ancient History' 'Angels' 'Angola'
 'Animal Fiction' 'Animals' 'Anime' 'Anthologies' 'Anthropology'
 'Anthropomorphic' 'Apocalyptic' 'Apple' 'Archaeology' 'Ar

In [32]:
few_genres = df["genres"].apply(lambda x: str(x).split("|")).explode().value_counts()
few_genres = few_genres[few_genres < 5]

print(few_genres)

genres
Hard Boiled             4
Cinderella              4
Fat                     4
Counter Culture         4
International Dev...    4
                       ..
Disability Studies      1
Peak Oil                1
Health Care             1
Read For College        1
Anthropomorphic         1
Name: count, Length: 237, dtype: int64


## Train model

General use case to train model using an algorithm sent by parameter, saving in a file for future use and returning the intended data for testing the model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
import joblib

def train_with(algorithm):
    # Declare X as global to ensure it is accessible within this function
    global X

    approach = input("Choose approach (1 for TF-IDF, 2 for Sentence Transformers): ")
    if approach == "2":
        # Load the precomputed embeddings from the .npy file
        X = np.load("book_desc_embeddings.npy")

    # Preprocess the genres to get the binary matrix Y
    Y, mlb = preprocess_genres(approach)

    # Split the dataset into training and testing sets (80% train, 20% test)
    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42
    )

    # Initialize the OneVsRestClassifier with the specified algorithm
    model = OneVsRestClassifier(algorithm)

    # Train the model on the training data
    model.fit(X_train, Y_train)
    print("Model trained successfully.")

    # Save the trained model to a file for later use
    joblib.dump(model, f"genre_classifier_{approach}_{algorithm.__class__.__name__}.joblib")
    # Save the MultiLabelBinarizer to a file for later use in decoding predictions
    joblib.dump(mlb, f"mlb_{approach}.joblib")


    return model, X_test, Y_test


### Train with Logistic regression

In [58]:
from sklearn.linear_model import LogisticRegression

model, X_test, Y_test = train_with(LogisticRegression(max_iter=2000))

c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 84 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 89 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 150 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 167 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multi

Model trained successfully.


conclusions from this 
- some genders are not present in any of the training data, so I assume they are part of the test data, so the model will not be able to guess a gender that it was not trained for, this will influence our tests
- probably some genders have just a few matching descriptions, so the model may be biased for the genres that has more descriptions

### Train with SVM
In theory has better accuracy between all non-NN algorithm, it maximizes margin between classes and handles well high dimensionality as the case of vectorized texts.

In [11]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

model, X_test, Y_test = train_with(CalibratedClassifierCV(LinearSVC()))

c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(0)

conclusions from this 
- As we have multiple labels without any training data we cannot use CalibratedClassifierCV to convert the distances to the decisions boundary to probabilities on the predictions, to work we would need to cleanup all labels for which we dont have descriptions on the training data and then apply the CalibratedClassifierCV

### Training with XGBoost
In theory, using a non-linear classifier that captures co-occurrence patterns, as for example, fantasy + adventure often occur together

In [12]:
from xgboost import XGBClassifier
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model, X_test, Y_test = train_with(
    XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        device=device
    )
)

Using device: cuda


c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 2 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 84 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 89 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 150 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multiclass.py:90: UserWarning: Label not 167 is present in all training examples.
  warnings.warn(
c:\Users\z004n7mh\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\multi

Model trained successfully.


### Test model

In [35]:
from sklearn.metrics import hamming_loss, accuracy_score, f1_score, precision_score, recall_score
model = joblib.load("genre_classifier_1_LogisticRegression.joblib")
Y_probs = model.predict_proba(X_test)

thresholds = (0.5, 0.4, 0.3, 0.2, 0.1)
values = [["Hamming Loss"], ["Accuracy"], ["Precision (micro)"], ["Recall (micro)"], ["F1 Score (micro)"]]
for threshold in thresholds:
    genre_pred = (Y_probs >= threshold).astype(int)

    hloss = hamming_loss(Y_test, genre_pred)
    acc = accuracy_score(Y_test, genre_pred)
    precision_micro = precision_score(Y_test, genre_pred, average='micro')
    recall_micro = recall_score(Y_test, genre_pred, average='micro')
    f1_micro = f1_score(Y_test, genre_pred, average='micro')

    values[0].append(f"{hloss:.4f}")
    values[1].append(f"{acc:.4f}")
    values[2].append(f"{precision_micro:.4f}")
    values[3].append(f"{recall_micro:.4f}")
    values[4].append(f"{f1_micro:.4f}")

results_df = pd.DataFrame(values, columns=["Metric"] + [str(th) for th in thresholds])
print(results_df)

              Metric     0.5     0.4     0.3     0.2     0.1
0       Hamming Loss  0.0047  0.0046  0.0047  0.0052  0.0078
1           Accuracy  0.0326  0.0353  0.0354  0.0208  0.0022
2  Precision (micro)  0.7985  0.7441  0.6750  0.5776  0.4148
3     Recall (micro)  0.3151  0.3796  0.4546  0.5444  0.6697
4   F1 Score (micro)  0.4519  0.5028  0.5433  0.5605  0.5123


conclusions from this 
- Hamming Loss = 0.0047: very low hamming loss is good, it means most individual labels are correctly predicted, but can be misleading as we have a lot of genres (864) and most of them are 0 por book
- Accuracy = 0.0277: very low accuracy is NOT good, but is usual in multi-label problems, because getting every label correct, knowing every book may have multiple genres, would be over exceptional
- Micro Precision = 0.7372: high micro precision is good, means a lot of predicted labels are actually correct, but may be, again, due to the fact we have a lot os labels and usually a book only have 2/3 genres, so all the other 800 would be 0
- Micro Recall = 0.3245: low micro recall es NOT good, this means it misses a lot of true genres, that is actually the goal
- Micro F1 Score = 04507: harmonic mean of micro-precision and micro-recall across all labels, will reflect the evolution of the model
- High precision + low recall + low F1 = the model is cautious, it predicts few labels, those it predicts are usually correct (high precision), but it misses many true genres (low recall)
- We can lower the threshold of the labels, but then there is a trade-off, precision decreases, recall increases
- We could even search the best threshold to apply for each gender, optimizing that way the f1 score

## Model usage

In [36]:
threshold = 0.5
approach = input("Choose approach (1 for TF-IDF, 2 for Sentence Transformers): ")
algorithm = input("Choose algorithm (1 for Logistic Regression, 2 for XGBoost): ")

algorithm_name = ""
if algorithm == "1":
    algorithm_name = "LogisticRegression"
elif algorithm == "2":
    algorithm_name = "XGBClassifier"

model = joblib.load(f"genre_classifier_{approach}_{algorithm_name}.joblib")
mlb = joblib.load(f"mlb_{approach}.joblib")
print("Model loaded successfully.")


def apply_preprocessing(texts):
    if approach == "1":
        return vectorizer.fit_transform([preprocess(clean_text(text)) for text in texts])
    elif approach == "2":
        return transformer.encode(
            [clean_text(text) for text in texts],
            normalize_embeddings=True,
            device=device
        )
    else:
        raise ValueError("Invalid approach selected. Please choose 1 or 2.")

texts = (
    "London, 1888. Beneath the fog-filled streets and flickering gas lamps, the city whispers with rumors of a series of strange disappearances. Each vanished person was last seen near the same narrow alley in Whitechapel—and each left behind an identical object: a small, intricately crafted pocket watch that always stops at exactly 3:17 a.m.",
    "In a distant future where advanced technology has unlocked portals to forgotten realms, humanity discovers a hidden universe filled with ancient magic and mythical creatures. When a young astrophysicist accidentally activates a long-dormant gateway orbiting Earth, she unleashes a mysterious force that begins merging science and sorcery.",
    "The Parker family just wanted a peaceful weekend getaway in the countryside. Instead, they end up renting an old mansion that the locals refuse to talk about. According to the listing, the house is “full of character.” What it doesn’t mention is that the character includes a grumpy ghost, a mischievous shadow creature, and a cursed doll that really hates loud children.",
    "Lena never believed in fate. As a determined art student trying to balance college, part-time jobs, and her dreams of becoming a painter, she’s convinced life is all about hard work—not destiny. Then she literally collides with Noah. Noah is a quiet but thoughtful engineering student who prefers books and late-night coffee over crowded parties. When Lena accidentally spills an entire cup of coffee over his carefully organized notes in a campus café, their first meeting is anything but romantic.",
    "For as long as they can remember, five friends have shared one strange thing in common: absolutely terrible luck.\nThere’s Jake, who once slipped on a banana peel… in a parking lot… where there were no bananas.\nMaya, whose phone always dies exactly when something important happens.\nLuis, who somehow sets off alarms in stores even when he doesn’t buy anything.\nSophie, whose GPS constantly sends the group to the wrong places.\nAnd Ben, who has broken six chairs in his life just by sitting down.\nDespite their ridiculous streak of misfortune, they decide to finally do something exciting together: a weekend road trip to attend a huge festival.",
    "Num reino onde a magia é proibida e as estrelas são vistas como presságios de desgraça, Elara esconde um segredo que pode mudar tudo: ela consegue ouvir o sussurro do céu.\nQuando conhece Kael, um misterioso viajante com um passado envolto em sombras, o destino de ambos entrelaça-se de forma inesperada. À medida que forças antigas despertam e uma guerra se aproxima, Elara e Kael são obrigados a confiar um no outro para sobreviver."
)
texts_vectors = apply_preprocessing(texts)

# Predict probabilities
genre_probs = model.predict_proba(texts_vectors)

for i, probs in enumerate(genre_probs):
    print(f"Book {i+1} genre probabilities:")
    for j, genre in enumerate(mlb.classes_):
        if probs[j] >= threshold:
            print(f"  {genre}: {probs[j]:.4f}")

Model loaded successfully.
Book 1 genre probabilities:
  Fantasy: 0.5111
  Fiction: 0.8768
  Science Fiction: 0.5403
Book 2 genre probabilities:
  Fantasy: 0.9778
  Fiction: 0.7160
  Science Fiction: 0.9398
Book 3 genre probabilities:
  Fiction: 0.8052
  Horror: 0.9225
Book 4 genre probabilities:
  Romance: 0.7736
Book 5 genre probabilities:
  Fiction: 0.6840
  Mystery: 0.5175
Book 6 genre probabilities:
  Fantasy: 0.9975
  Fiction: 0.6147
  Magic: 0.6290
  Young Adult: 0.6794
